In [ ]:
# Colab: Persistent Security Lab Workspace (save tools + outputs to Google Drive)
# RUN ONLY against targets you own or have permission to test.

# -------------------------
# 0) Imports & Drive mount
# -------------------------
import os, sys, subprocess, time, shutil
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')   # authorize once
DRIVE_ROOT = "/content/drive/MyDrive/sec-lab"   # change if you like
os.makedirs(DRIVE_ROOT, exist_ok=True)

# -------------------------
# 1) Paths (persistent)
# -------------------------
TOOLS_DIR = os.path.join(DRIVE_ROOT, "tools")           # git repos cached here
RESULTS_DIR = os.path.join(DRIVE_ROOT, "scan-results")  # outputs saved here
CACHE_DIR = os.path.join(DRIVE_ROOT, "cache")           # any other cached files
os.makedirs(TOOLS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print("Workspace:", DRIVE_ROOT)
print("Tools cache:", TOOLS_DIR)
print("Results:", RESULTS_DIR)

# -------------------------
# 2) Utility: run and save output
# -------------------------
def run_and_save(cmd, outpath, timeout=None):
    """Run shell command and write stdout+stderr to outpath file (streaming)."""
    print(f"\n[RUN] {cmd}\n-> saving to {outpath}\n")
    with open(outpath, "wb") as fout:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        try:
            for line in proc.stdout:
                fout.write(line)
                fout.flush()
                # also print live to notebook
                try:
                    print(line.decode('utf-8', errors='ignore'), end='')
                except:
                    pass
            proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            proc.terminate()
            fout.write(b"\n[Timed out]\n")
    print(f"\n[Saved] {outpath}\n")
    return outpath

# -------------------------
# 3) Ensure basic system deps (installs per session)
# -------------------------
def ensure_system_tools():
    """Install system-level packages that can't persist across Colab VM restarts."""
    required = ["nmap", "wget", "curl", "unzip", "git", "python3-pip", "default-jre"]
    # Check and install if missing (apt-get is idempotent)
    install_list = []
    for pkg in required:
        which_cmd = f"which {pkg.split('-')[0]} >/dev/null 2>&1 && echo OK || echo MISSING"
        r = subprocess.run(which_cmd, shell=True, capture_output=True, text=True)
        if "MISSING" in r.stdout:
            install_list.append(pkg)
    if install_list:
        print("Installing missing system packages:", install_list)
        # Apt-get update then install
        subprocess.run("apt-get update -qq", shell=True)
        subprocess.run(f"apt-get install -y -qq {' '.join(install_list)}", shell=True)
    else:
        print("All system packages present.")
    # ensure pip packages
    subprocess.run("pip install -q requests", shell=True)

# Run once per session
ensure_system_tools()

# -------------------------
# 4) Cache/clone tool repos into Drive
# -------------------------
# (We clone into Drive so repo files persist across sessions)
REPOS = {
    "sqlmap": "https://github.com/sqlmapproject/sqlmap.git",
    "nikto": "https://github.com/sullo/nikto.git",
    "wfuzz": "https://github.com/xmendez/wfuzz.git",       # optional
    "gobuster": "https://github.com/OJ/gobuster.git"       # optional (requires go build)
}

def ensure_tool_repos():
    """Clone or pull the repos into TOOLS_DIR on Drive."""
    for name, repo in REPOS.items():
        repo_path = os.path.join(TOOLS_DIR, name)
        if os.path.exists(repo_path):
            print(f"[CACHE] Updating {name}")
            subprocess.run(f"cd {repo_path} && git pull --quiet", shell=True)
        else:
            print(f"[CLONE] {name}")
            subprocess.run(f"git clone --depth 1 {repo} {repo_path}", shell=True)

ensure_tool_repos()

# -------------------------
# 5) Helper wrappers for popular tools
# -------------------------
def run_nmap(target, extra_args="--top-ports 1000 -sV -T4", outfolder=None):
    ts = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    safe_target = target.replace(":", "_").replace("/", "_").replace(".", "_")
    outfolder = outfolder or os.path.join(RESULTS_DIR, safe_target, ts)
    os.makedirs(outfolder, exist_ok=True)
    outtxt = os.path.join(outfolder, "nmap.txt")
    outxml = os.path.join(outfolder, "nmap.xml")
    cmd = f"nmap {extra_args} -oN {outtxt} -oX {outxml} {target}"
    run_and_save(cmd, os.path.join(outfolder, "nmap-run.log"))
    return outfolder

def run_nikto(target, outfolder=None):
    ts = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    safe_target = target.replace(":", "_").replace("/", "_").replace(".", "_")
    outfolder = outfolder or os.path.join(RESULTS_DIR, safe_target, ts)
    os.makedirs(outfolder, exist_ok=True)
    outtxt = os.path.join(outfolder, "nikto.txt")
    # Nikto repository contains a nikto.pl, but some systems provide nikto binary via apt
    nikto_bin = shutil.which("nikto") or os.path.join(TOOLS_DIR, "nikto", "nikto.pl")
    cmd = f"perl {nikto_bin} -h {target} -output {outtxt}" if nikto_bin.endswith(".pl") else f"{nikto_bin} -h {target} -o {outtxt}"
    run_and_save(cmd, os.path.join(outfolder, "nikto-run.log"))
    return outfolder

def run_sqlmap(target_url, outfolder=None, extra_args="--batch --level=2"):
    # target_url must be the full URL of a vulnerable param, e.g. http://10.10.10.10/vuln.php?id=1
    ts = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    safe_target = target_url.replace("://", "_").replace("/", "_").replace("?", "_").replace("&", "_")
    outfolder = outfolder or os.path.join(RESULTS_DIR, safe_target, ts)
    os.makedirs(outfolder, exist_ok=True)
    sqlmap_dir = os.path.join(TOOLS_DIR, "sqlmap")
    sqlmap_py = os.path.join(sqlmap_dir, "sqlmap.py")
    if not os.path.exists(sqlmap_py):
        print("[ERROR] sqlmap not found in tools cache. Please run ensure_tool_repos().")
        return outfolder
    outtxt = os.path.join(outfolder, "sqlmap.txt")
    cmd = f"python3 {sqlmap_py} -u \"{target_url}\" {extra_args} -o --output-dir={outfolder}"
    # sqlmap can produce many files; we also capture wrapper log
    run_and_save(cmd, os.path.join(outfolder, "sqlmap-run.log"), timeout=3600)  # timeout adjustable
    return outfolder

# -------------------------
# 6) Simple interactive CLI helper for convenience
# -------------------------
def interactive_cli():
    print("\n=== sec-lab CLI ===")
    print("Available commands:")
    print("  1) nmap <target> [extra_args]")
    print("  2) nikto <host>          (e.g. nikto 10.10.10.10 or nikto 10.10.10.10:8080)")
    print("  3) sqlmap <url>          (target URL with vulnerable param)")
    print("  4) refresh-tools         (git pull cached repos)")
    print("  5) exit")
    while True:
        try:
            cmd = input("sec-lab> ").strip()
        except EOFError:
            break
        if not cmd:
            continue
        parts = cmd.split()
        if parts[0] == "nmap" and len(parts) >= 2:
            target = parts[1]
            extra = " ".join(parts[2:]) if len(parts) > 2 else "--top-ports 1000 -sV -T4"
            out = run_nmap(target, extra_args=extra)
            print("Saved outputs to:", out)
        elif parts[0] == "nikto" and len(parts) == 2:
            out = run_nikto(parts[1])
            print("Saved outputs to:", out)
        elif parts[0] == "sqlmap" and len(parts) == 2:
            out = run_sqlmap(parts[1])
            print("Saved outputs to:", out)
        elif parts[0] == "refresh-tools":
            ensure_tool_repos()
        elif parts[0] == "exit":
            break
        else:
            print("Unknown command or wrong args. Try again.")

# -------------------------
# 7) Usage demo + notes
# -------------------------
print("\nSetup complete. You can now run scans and keep results in Drive.")
print("To run interactive CLI: call interactive_cli()")
print("Example quick runs:")
print("  run_nmap('127.0.0.1')    # change to your lab VM IP")
print("  run_nikto('10.10.10.10:80')")
print("  run_sqlmap('http://10.10.10.10/vuln.php?id=1')")

# Uncomment to start CLI immediately:
interactive_cli()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workspace: /content/drive/MyDrive/sec-lab
Tools cache: /content/drive/MyDrive/sec-lab/tools
Results: /content/drive/MyDrive/sec-lab/scan-results
Installing missing system packages: ['default-jre']
[CACHE] Updating sqlmap
[CACHE] Updating nikto
[CACHE] Updating wfuzz
[CACHE] Updating gobuster

Setup complete. You can now run scans and keep results in Drive.
To run interactive CLI: call interactive_cli()
Example quick runs:
  run_nmap('127.0.0.1')    # change to your lab VM IP
  run_nikto('10.10.10.10:80')
  run_sqlmap('http://10.10.10.10/vuln.php?id=1')

=== sec-lab CLI ===
Available commands:
  1) nmap <target> [extra_args]
  2) nikto <host>          (e.g. nikto 10.10.10.10 or nikto 10.10.10.10:8080)
  3) sqlmap <url>          (target URL with vulnerable param)
  4) refresh-tools         (git pull cached repos)
  5) exit
sec-lab> sqlmap seagis.ac.lk


/tmp/ipython-input-3259246108.py:130: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y%m%d-%H%M%S")



[RUN] python3 /content/drive/MyDrive/sec-lab/tools/sqlmap/sqlmap.py -u "seagis.ac.lk" --batch --level=2 -o --output-dir=/content/drive/MyDrive/sec-lab/scan-results/seagis.ac.lk/20250924-174713
-> saving to /content/drive/MyDrive/sec-lab/scan-results/seagis.ac.lk/20250924-174713/sqlmap-run.log

        ___
       __H__
 ___ ___["]_____ ___ ___  {1.9.9.4#dev}
|_ -| . [,]     | .'| . |
|___|_  ["]_|_|_|__,|  _|
      |_|V...       |_|   https://sqlmap.org

[!] legal disclaimer: Usage of sqlmap for attacking targets without prior mutual consent is illegal. It is the end user's responsibility to obey all applicable local, state and federal laws. Developers assume no liability and are not responsible for any misuse or damage caused by this program

[*] starting @ 17:47:15 /2025-09-24/

>[17:47:15] [WARNING] using '/content/drive/MyDrive/sec-lab/scan-results/seagis.ac.lk/20250924-174713' as the output directory
[1/1] URL:
GET http://seagis.ac.lk
do you want to test this URL? [Y/n/q]
> Y
[17